# Local-currency non-resident bonds (`PV_LC_NR`)

`LocalCurrencyNonResidentInstrument` mirrors the hidden `PV_LC_NR1/2/3` sheets:
LC vintages + FX(pa)/FX(eop) → USD Interest / Amortization / PV / Stock.

Sibling of `PresentValueInstrument` (USD / `PV_Base`), not an LC flag on that class.

See `docs/03-pv-instruments.qmd`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

from lic_dsf.pv import (
    LocalCurrencyNonResidentInstrument,
    PVPortfolio,
    load_lc_nr_instruments_from_workbook,
)

pd.set_option("display.max_columns", 12)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
WORKBOOK

## Load the three Input 5 / PV_LC_NR tenors

In [ ]:
instruments = load_lc_nr_instruments_from_workbook(WORKBOOK)
catalog = pd.DataFrame(
    [
        {
            "name": i.name,
            "grace": i.grace,
            "maturity": i.maturity,
            "discount": i.discount_rate,
            "disbursement_lc_sum": sum(i.disbursements_lc),
            "n_years": len(i.disbursements_lc),
        }
        for i in instruments
    ]
)
catalog

## Output panel (Ext_Debt-facing USD metrics)

In [ ]:
short = next(i for i in instruments if "1 to 3" in i.name)
external = short.external()
external.loc[
    [
        "Interest",
        "Amortization",
        "PV of debt",
        "Stock of new forex debt (in USD)",
    ],
    :10,
]

## Optional: fold into `PVPortfolio`

`external()` uses canonical Interest / Amortization / PV / Stock rows, so LC-NR
instruments can sit in the same portfolio as USD `PresentValueInstrument`s.

In [ ]:
portfolio = PVPortfolio(tuple(instruments))
portfolio.interest().iloc[:, :8]